In [67]:
import os
import sys

# Standard interactive replacement for the 'parent directory' hack
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)

import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

from PPRCalculator import PPRCalculator
from ModelData import ModelData

In [103]:
def apply_ecopath_defaults(df, DC, det_fate=None, zero_catch=False, zero_biomass_accum=False, default_gs=False):
    """
    Applies Ecopath defaults and ensures flows are synced with ratios.
    Assumes p, q, ee, and catch are given. assumes p, q not given for detritus group.
        calculates M0 = p*(1-ee)
    Assumes gs is 0.2 for regular groups and 0 for pp, det, diet_import.
        calculates egestion = q*gs.
    Assumes biomas_accum, net_migration are 0 where currently nan.
    Calculates once cell missing from p, q, egestion, respiration.
    Calculates predation = Z.sum(axis=1).
        Calculates once cell missing from M0, predation, catch, biomass_accum, net_migration.
    """
    df = df.sort_index(ascending=False)
    is_regular = df["trophic_info"] == "Regular"
    is_import = df["trophic_info"] == "Import"
    is_det = df["trophic_info"] == "DET"
    is_pp = df["trophic_info"] == "PP"

    # original_nas = df.isna()

    # cols with constant value of 0 or 1:
    cols_where_default_is_zero = ['detritus_import', 'immigration', 'emigration', 'net_migration']
    if zero_catch:
        cols_where_default_is_zero.append('catch')
    if zero_biomass_accum:
        cols_where_default_is_zero.append('biomass_accum')
    fill_dict = {col: 0 for col in cols_where_default_is_zero}
    df = df.fillna(fill_dict)
    df.loc[:, 'biomass'] = df['biomass'].fillna(1)
    # set biomass_accum for det to nan anyway:
    df.loc[is_det, 'biomass_accum'] = np.nan

    # get Z which is correct for all regular groups:
    Z = DC.mul(df['q'].fillna(0), axis='index')
    Z.loc[is_det | is_pp | is_import, :] = 0
    Z = Z.sort_index(ascending=False).sort_index(ascending=False, axis=1)

    # set default values for non-regular groups:
    df.loc[is_pp | is_import | is_det, ['egestion', 'respiration']] = 0
    df.loc[is_import | is_det, 'M0'] = 0
    df.loc[is_import | is_det, 'ee'] = 1

    # fillna for gs and egestion:
    df.loc[~is_regular, 'gs'] = df.loc[~is_regular, 'gs'].fillna(0)
    if default_gs:
        df.loc[is_regular, 'gs'] = df.loc[is_regular, 'gs'].fillna(0.2)

    # Sync gs -> egestion
    mask_sync = df['egestion'].isna() & df['q'].notna() & df['gs'].notna()
    df.loc[mask_sync, 'egestion'] = df['q'] * df['gs']

    # Sync ee -> M0
    mask_sync = df['M0'].isna() & df["p"].notna() & df["ee"].notna()
    df.loc[mask_sync, 'M0'] = df["p"] * (1 - df["ee"])

    # q for detritus:
    df['flow_to_det'] = df['M0'] + df['egestion']
    if det_fate is not None:
        detritus_inflows = det_fate.mul(df['flow_to_det'], axis='index').sum(axis=0)
        df.loc[is_det, 'q'] = df.loc[is_det, 'q'].fillna(detritus_inflows)
    else:
        df.loc[is_det, 'q'] = df.loc[is_det, 'q'].fillna(df['M0'].sum() + df['egestion'].sum())  # set q of detritus as M0.sum() + egestion.sum()

    # predation:
    df.loc[:, 'predation'] = Z.sum(axis=0)

    # for non-regular groups, egestion and respiration are nan so q = p:
    df.loc[is_det, 'p'] = df['q']
    df.loc[is_pp | is_import, 'q'] = df['p']
    
    # pb and qb:
    df.loc[:, 'pb'] = (df['p'] / df['biomass']).fillna(0)
    df.loc[:, 'qb'] = (df['q'] / df['biomass']).fillna(0)

    def _solve_linear_equation(df, eq_cols, signs):
        # Identify rows where exactly ONE variable is missing (otherwise it's unsolvable this way)
        solvable_mask = df[eq_cols].isna().sum(axis=1) == 1

        # Calculate the net sum of all known terms (Pandas ignores NaNs by default in .sum)
        balance_sum = (df[eq_cols] * signs).sum(axis=1, skipna=True)

        # Fill the missing cells
        for col in eq_cols:
            # Target rows where THIS column is the missing one, and the equation is solvable
            target_cells = df[col].isna() & solvable_mask
            
            # The missing value is the negative balance_sum divided by the column's sign
            df.loc[target_cells, col] = -balance_sum[target_cells] / signs[col]
        
        return df
    
    # consumption equation:
    eq_cols = ['q', 'p', 'respiration', 'egestion']
    signs = pd.Series({
        'q': 1, 
        'p': -1, 
        'respiration': -1, 
        'egestion': -1, 
    })
    df = _solve_linear_equation(df, eq_cols, signs)
    
    # production equation:
    eq_cols = ['p', 'M0', 'catch', 'predation', 'net_migration', 'biomass_accum']
    signs = pd.Series({
        'p': 1, 
        'M0': -1, 
        'catch': -1, 
        'predation': -1, 
        'net_migration': -1, 
        'biomass_accum': -1
    })
    df = _solve_linear_equation(df, eq_cols, signs)

    # finalize ratios:
    df.loc[is_regular, 'ee'] = 1 - (df['M0'] / df['p'])
    df.loc[is_regular, 'gs'] = (df['egestion'] / df['q'])
    df['ge'] = (df['p'] / df['q'])
    df['flow_to_det'] = df['flow_to_det'].fillna(df['M0'] + df['egestion'])

    return df

In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize

def apply_lim(df, weight_flow=1.0, weight_guess=0.0):
    """
    Uses Linear Inverse Modeling (SLSQP) to fill in missing mass-balance 
    variables by dynamically registering active constraints and coordinating guesses.
    """
    df_out = df.copy()

    def _finalize_outputs(df_final):
        # EE
        df_final.loc[:, 'ee'] = np.where(
            df_final.loc[:, 'p'] != 0, 
            1 - (df_final.loc[:, 'M0'] / df_final.loc[:, 'p']), 
            0
        )
        # GS
        df_final.loc[:, 'gs'] = np.where(
            df_final.loc[:, 'q'] != 0, 
            df_final.loc[:, 'egestion'] / df_final.loc[:, 'q'], 
            0
        )
        # GE
        df_final['ge'] = np.where(df_final['q'] != 0, df_final['p'] / df_final['q'], 0)
        
        # Flow to Detritus
        df_final['flow_to_det'] = df_final['M0'] + df_final['egestion']
        
        return df_final

    cols_to_solve = ['q', 'p', 'respiration', 'egestion', 'M0', 'biomass_accum']
    records = df_out.to_dict('index')
    
    for idx, row in records.items():
        missing_cols = [c for c in cols_to_solve if pd.isna(row[c])]
        if not missing_cols:
            continue
            
        c_val = row['catch'] if pd.notna(row['catch']) else 0
        pr_val = row['predation'] if pd.notna(row['predation']) else 0
        nm_val = row['net_migration'] if pd.notna(row['net_migration']) else 0

        # --- 1. Coordinated Guess Variables ---
        q_guess = row['q'] if pd.notna(row['q']) else (row['p'] * 10 if pd.notna(row['p']) else 1.0)
        p_guess = row['p'] if pd.notna(row['p']) else (q_guess * 0.1)
        
        # Smart adaptation: If both P and Q are known, don't overshoot the available pool
        if pd.notna(row['q']) and pd.notna(row['p']):
            rem_pool = max(0.0, row['q'] - row['p'])
            resp_guess = row['respiration'] if pd.notna(row['respiration']) else (rem_pool * (0.7 / 0.9))
            egest_guess = row['egestion'] if pd.notna(row['egestion']) else (rem_pool * (0.2 / 0.9))
        else:
            resp_guess = row['respiration'] if pd.notna(row['respiration']) else (q_guess * 0.7)
            egest_guess = row['egestion'] if pd.notna(row['egestion']) else (q_guess * 0.2)
            
        m0_guess = row['M0'] if pd.notna(row['M0']) else (q_guess * 0.1)
        ba_guess = row['biomass_accum'] if pd.notna(row['biomass_accum']) else (p_guess - m0_guess - c_val - pr_val - nm_val)

        # Assemble variables and bounds
        x0 = []
        bounds = []
        for col in missing_cols:
            if col == 'q': x0.append(q_guess); bounds.append((0, None))
            elif col == 'p': x0.append(p_guess); bounds.append((0, None))
            elif col == 'respiration': x0.append(resp_guess); bounds.append((0, None))
            elif col == 'egestion': x0.append(egest_guess); bounds.append((0, None))
            elif col == 'M0': x0.append(m0_guess); bounds.append((0, None))
            elif col == 'biomass_accum': x0.append(ba_guess); bounds.append((None, None))

        x0_arr = np.array(x0)
        scale = np.where(np.abs(x0_arr) > 0.01, np.abs(x0_arr), 1.0)
        
        # --- 2. High-Speed Mapping Indices ---
        idx_q = missing_cols.index('q') if 'q' in missing_cols else -1
        idx_p = missing_cols.index('p') if 'p' in missing_cols else -1
        idx_r = missing_cols.index('respiration') if 'respiration' in missing_cols else -1
        idx_e = missing_cols.index('egestion') if 'egestion' in missing_cols else -1
        idx_m0 = missing_cols.index('M0') if 'M0' in missing_cols else -1
        idx_ba = missing_cols.index('biomass_accum') if 'biomass_accum' in missing_cols else -1

        # --- 3. Objective Function ---
        def objective(x):
            # Penalty A: Flow Penalty (Parsimony / Minimum Total Flow)
            # Minimizes the total magnitude of the actively solved flows
            flow_pen = np.sum(np.abs(x) / scale)
            
            # Penalty B: Guess Penalty (Biological Anchoring)
            # Minimizes squared deviation from your smart biological guesses (x0)
            guess_pen = np.sum(((x - x0_arr) / scale) ** 2)
            
            return (weight_flow * flow_pen) + (weight_guess * guess_pen)

        # --- 4. Dynamic Constraint Registration ---
        constraints = []
        
        # Consumption Balance (Only if Q, P, R, or U are being optimized)
        if any(c in missing_cols for c in ['q', 'p', 'respiration', 'egestion']):
            def eq_consumption(x):
                q_curr = x[idx_q] if idx_q >= 0 else q_guess
                p_curr = x[idx_p] if idx_p >= 0 else p_guess
                r_curr = x[idx_r] if idx_r >= 0 else resp_guess
                e_curr = x[idx_e] if idx_e >= 0 else egest_guess
                return q_curr - (p_curr + r_curr + e_curr)
            constraints.append({'type': 'eq', 'fun': eq_consumption})
            
        # Production Balance (Only if P, M0, or BA are being optimized)
        if any(c in missing_cols for c in ['p', 'M0', 'biomass_accum']):
            def eq_production(x):
                p_curr = x[idx_p] if idx_p >= 0 else p_guess
                m0_curr = x[idx_m0] if idx_m0 >= 0 else m0_guess
                ba_curr = x[idx_ba] if idx_ba >= 0 else ba_guess
                return p_curr - (m0_curr + c_val + pr_val + nm_val + ba_curr)
            constraints.append({'type': 'eq', 'fun': eq_production})

        # EE bounds (Only if P or M0 can actually change)
        if 'p' in missing_cols or 'M0' in missing_cols:
            # Upper Bound: EE <= 0.95 (Equivalent to: M0 - 0.05 * P >= 0)
            def ineq_ee_upper(x):
                p_curr = x[idx_p] if idx_p >= 0 else p_guess
                m0_curr = x[idx_m0] if idx_m0 >= 0 else m0_guess
                return m0_curr - (0.05 * p_curr)
            constraints.append({'type': 'ineq', 'fun': ineq_ee_upper})
            
            # Lower Bound: EE >= 0.0 (Equivalent to: P - M0 >= 0)
            def ineq_ee_lower(x):
                p_curr = x[idx_p] if idx_p >= 0 else p_guess
                m0_curr = x[idx_m0] if idx_m0 >= 0 else m0_guess
                return p_curr - m0_curr
            constraints.append({'type': 'ineq', 'fun': ineq_ee_lower})
            
        # GS Upper limit (Only if Q or Egestion can actually change)
        if 'q' in missing_cols or 'egestion' in missing_cols:
            # Enforce GS <= 0.35
            def ineq_gs_upper(x):
                q_curr = x[idx_q] if idx_q >= 0 else q_guess
                e_curr = x[idx_e] if idx_e >= 0 else egest_guess
                return 0.35 * q_curr - e_curr
            constraints.append({'type': 'ineq', 'fun': ineq_gs_upper})

            # Enforce GS >= 0.10
            def ineq_gs_lower(x):
                q_curr = x[idx_q] if idx_q >= 0 else q_guess
                e_curr = x[idx_e] if idx_e >= 0 else egest_guess
                return e_curr - (0.10 * q_curr)
            constraints.append({'type': 'ineq', 'fun': ineq_gs_lower})

        # --- 5. Run Optimizer ---
        res = minimize(
            objective, 
            x0_arr, 
            method='SLSQP', 
            bounds=bounds, 
            constraints=constraints,
            tol=1e-5,
            options={'maxiter': 1500, 'ftol': 1e-5}
        )

        if not res.success:
            print(f"LIM failed for group '{idx}': {res.message}")
        else:
            for i, col in enumerate(missing_cols):
                df_out.loc[idx, col] = res.x[i]

    return _finalize_outputs(df_out)

In [139]:
model = PPRCalculator(459)
model.get_groups_df().round(5)[['group_name', 'trophic_info', 'tl', 'ee', 'p', 'q', 'biomass_accum', 'net_migration', 'M0', 'predation', 'catch', 'gs', 'egestion', 'respiration']]
model.get_groups_df().round(5)

,group_name,trophic_info,tl,ge,ee,catch,biomass,pb,qb,p,...,M0,gs,egestion,respiration,biomass_accum,emigration,immigration,net_migration,flow_to_det,detritus_import
group_seq,,,,,,,,,,,,,,,,,,,,,
32,diet_import,Import,1.00000,1.00000,1.00000,0.00000,NaN,NaN,NaN,84.56694,...,0.00000,0.0,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,NaN
31,Detritus,DET,1.00000,1.00000,0.00000,0.00000,15.07500,NaN,NaN,1642.61581,...,0.00000,0.0,0.00000,0.00000,491.83744,0.0,0.0,0.0,NaN,0.0
30,Phytoplankton,PP,1.00000,1.00000,0.84209,0.00000,32.00000,42.800,0.00,1369.59993,...,216.27271,0.0,0.00000,0.00000,0.00000,0.0,0.0,0.0,216.27271,0.0
29,Large zooplankton,Regular,3.00597,0.30000,0.15243,0.00000,9.63600,8.700,29.00,83.83320,...,71.05464,0.4,111.77760,83.83320,0.00000,0.0,0.0,0.0,182.83224,0.0
28,Small zooplankton,Regular,2.00301,0.29983,0.66559,0.00000,40.00000,17.300,57.70,692.00000,...,231.41099,0.4,923.20000,692.80000,0.00000,0.0,0.0,0.0,1154.61099,0.0
27,Large squids,Regular,4.39837,0.29004,0.35927,0.00001,0.17709,4.600,15.86,0.81460,...,0.52194,0.2,0.56172,1.43229,0.00000,0.0,0.0,0.0,1.08366,0.0
26,Small squids,Regular,3.86046,0.30005,0.95000,0.00000,1.15744,5.500,18.33,6.36594,...,0.31830,0.2,4.24319,10.60681,0.00000,0.0,0.0,0.0,4.56148,0.0
25,Other turtles,Regular,3.50449,0.04286,0.99000,0.00003,0.00077,0.150,3.50,0.00012,...,0.00000,0.2,0.00054,0.00203,0.00000,0.0,0.0,0.0,0.00054,0.0
24,Leatherback turtle,Regular,3.98591,0.04286,0.99000,0.00001,0.00057,0.150,3.50,0.00009,...,0.00000,0.2,0.00040,0.00151,0.00000,0.0,0.0,0.0,0.00040,0.0


In [147]:
groups_df = model.get_groups_df().copy()
groups_df.round(5)

,group_name,trophic_info,tl,ge,ee,catch,biomass,pb,qb,p,...,M0,gs,egestion,respiration,biomass_accum,emigration,immigration,net_migration,flow_to_det,detritus_import
group_seq,,,,,,,,,,,,,,,,,,,,,
32,diet_import,Import,1.00000,1.00000,1.00000,0.00000,NaN,NaN,NaN,84.56694,...,0.00000,0.0,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,NaN
31,Detritus,DET,1.00000,1.00000,0.00000,0.00000,15.07500,NaN,NaN,1642.61581,...,0.00000,0.0,0.00000,0.00000,491.83744,0.0,0.0,0.0,NaN,0.0
30,Phytoplankton,PP,1.00000,1.00000,0.84209,0.00000,32.00000,42.800,0.00,1369.59993,...,216.27271,0.0,0.00000,0.00000,0.00000,0.0,0.0,0.0,216.27271,0.0
29,Large zooplankton,Regular,3.00597,0.30000,0.15243,0.00000,9.63600,8.700,29.00,83.83320,...,71.05464,0.4,111.77760,83.83320,0.00000,0.0,0.0,0.0,182.83224,0.0
28,Small zooplankton,Regular,2.00301,0.29983,0.66559,0.00000,40.00000,17.300,57.70,692.00000,...,231.41099,0.4,923.20000,692.80000,0.00000,0.0,0.0,0.0,1154.61099,0.0
27,Large squids,Regular,4.39837,0.29004,0.35927,0.00001,0.17709,4.600,15.86,0.81460,...,0.52194,0.2,0.56172,1.43229,0.00000,0.0,0.0,0.0,1.08366,0.0
26,Small squids,Regular,3.86046,0.30005,0.95000,0.00000,1.15744,5.500,18.33,6.36594,...,0.31830,0.2,4.24319,10.60681,0.00000,0.0,0.0,0.0,4.56148,0.0
25,Other turtles,Regular,3.50449,0.04286,0.99000,0.00003,0.00077,0.150,3.50,0.00012,...,0.00000,0.2,0.00054,0.00203,0.00000,0.0,0.0,0.0,0.00054,0.0
24,Leatherback turtle,Regular,3.98591,0.04286,0.99000,0.00001,0.00057,0.150,3.50,0.00009,...,0.00000,0.2,0.00040,0.00151,0.00000,0.0,0.0,0.0,0.00040,0.0


In [154]:
# groups_df[['egestion', 'respiration', 'biomass_accum', 'emigration', 'immigration', 'net_migration', 'M0']] = np.nan
groups_df[['M0', 'respiration', 'egestion', 'biomass_accum', 'gs', 'ee']] = np.nan
# groups_df[['M0', 'respiration', 'egestion', 'biomass_accum']] = np.nan

groups_df = apply_ecopath_defaults(groups_df, model.get_DC(), det_fate=model._det_fate, zero_biomass_accum=False, default_gs=False).round(6)
groups_df

,group_name,trophic_info,tl,ge,ee,catch,biomass,pb,qb,p,...,M0,gs,egestion,respiration,biomass_accum,emigration,immigration,net_migration,flow_to_det,detritus_import
group_seq,,,,,,,,,,,,,,,,,,,,,
32,diet_import,Import,1.000000,1.000000,1.0,0.000000,1.000000,84.566940,84.566940,84.566940,...,0.0,0.0,0.0,0.0,-0.000000,0.0,0.0,0.0,0.0,0.0
31,Detritus,DET,1.000000,1.000000,1.0,0.000000,15.075000,108.962906,108.962906,1642.615813,...,0.0,0.0,0.0,0.0,491.837442,0.0,0.0,0.0,0.0,0.0
30,Phytoplankton,PP,1.000000,1.000000,NaN,0.000000,32.000000,42.799998,42.799998,1369.599930,...,NaN,0.0,0.0,0.0,NaN,0.0,0.0,0.0,NaN,0.0
29,Large zooplankton,Regular,3.005970,0.300000,NaN,0.000000,9.636000,8.700000,29.000000,83.833200,...,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,0.0
28,Small zooplankton,Regular,2.003012,0.299827,NaN,0.000000,40.000000,17.300000,57.700000,692.000000,...,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,0.0
27,Large squids,Regular,4.398368,0.290038,NaN,0.000007,0.177088,4.600001,15.860002,0.814605,...,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,0.0
26,Small squids,Regular,3.860465,0.300055,NaN,0.000000,1.157443,5.500000,18.329999,6.365936,...,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,0.0
25,Other turtles,Regular,3.504491,0.042846,NaN,0.000035,0.000767,0.149935,3.499348,0.000115,...,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,0.0
24,Leatherback turtle,Regular,3.985911,0.042649,NaN,0.000006,0.000569,0.149385,3.502636,0.000085,...,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,0.0


In [158]:
apply_lim(groups_df.copy(), weight_flow=1.0, weight_guess=0.0).round(5)

,group_name,trophic_info,tl,ge,ee,catch,biomass,pb,qb,p,...,M0,gs,egestion,respiration,biomass_accum,emigration,immigration,net_migration,flow_to_det,detritus_import
group_seq,,,,,,,,,,,,,,,,,,,,,
32,diet_import,Import,1.00000,1.00000,1.00000,0.00000,1.00000,84.56694,84.56694,84.56694,...,0.00000,0.00000,0.00000,0.00000,-0.00000,0.0,0.0,0.0,0.00000,0.0
31,Detritus,DET,1.00000,1.00000,1.00000,0.00000,15.07500,108.96291,108.96291,1642.61581,...,0.00000,0.00000,0.00000,0.00000,491.83744,0.0,0.0,0.0,0.00000,0.0
30,Phytoplankton,PP,1.00000,1.00000,0.84209,0.00000,32.00000,42.80000,42.80000,1369.59993,...,216.27101,0.00000,0.00000,0.00000,0.00171,0.0,0.0,0.0,216.27101,0.0
29,Large zooplankton,Regular,3.00597,0.30000,0.95000,0.00000,9.63600,8.70000,29.00000,83.83320,...,4.19166,0.10000,27.94440,167.66640,66.86298,0.0,0.0,0.0,32.13606,0.0
28,Small zooplankton,Regular,2.00301,0.29983,0.66559,0.00000,40.00000,17.30000,57.70000,692.00000,...,231.41104,0.15559,359.10888,1256.89112,0.00000,0.0,0.0,0.0,590.51993,0.0
27,Large squids,Regular,4.39837,0.29004,0.35928,0.00001,0.17709,4.60000,15.86000,0.81460,...,0.52193,0.12195,0.34251,1.65150,0.00001,0.0,0.0,0.0,0.86444,0.0
26,Small squids,Regular,3.86046,0.30005,0.95000,0.00000,1.15744,5.50000,18.33000,6.36594,...,0.31830,0.10000,2.12159,12.72840,-0.00000,0.0,0.0,0.0,2.43989,0.0
25,Other turtles,Regular,3.50449,0.04285,0.95000,0.00003,0.00077,0.14994,3.49935,0.00012,...,0.00001,0.21270,0.00057,0.00200,-0.00000,0.0,0.0,0.0,0.00058,0.0
24,Leatherback turtle,Regular,3.98591,0.04265,0.95000,0.00001,0.00057,0.14938,3.50264,0.00008,...,0.00000,0.21274,0.00042,0.00148,-0.00000,0.0,0.0,0.0,0.00043,0.0


In [160]:
apply_lim(groups_df.copy(), weight_flow=0.0, weight_guess=1.0).round(5)

,group_name,trophic_info,tl,ge,ee,catch,biomass,pb,qb,p,...,M0,gs,egestion,respiration,biomass_accum,emigration,immigration,net_migration,flow_to_det,detritus_import
group_seq,,,,,,,,,,,,,,,,,,,,,
32,diet_import,Import,1.00000,1.00000,1.00000,0.00000,1.00000,84.56694,84.56694,84.56694,...,0.00000,0.00000,0.00000,0.00000,-0.00000,0.0,0.0,0.0,0.00000,0.0
31,Detritus,DET,1.00000,1.00000,1.00000,0.00000,15.07500,108.96291,108.96291,1642.61581,...,0.00000,0.00000,0.00000,0.00000,491.83744,0.0,0.0,0.0,0.00000,0.0
30,Phytoplankton,PP,1.00000,1.00000,0.90000,0.00000,32.00000,42.80000,42.80000,1369.59993,...,136.95999,0.00000,0.00000,0.00000,79.31272,0.0,0.0,0.0,136.95999,0.0
29,Large zooplankton,Regular,3.00597,0.30000,0.66667,0.00000,9.63600,8.70000,29.00000,83.83320,...,27.94440,0.15556,43.46907,152.14173,43.11024,0.0,0.0,0.0,71.41347,0.0
28,Small zooplankton,Regular,2.00301,0.29983,0.66647,0.00000,40.00000,17.30000,57.70000,692.00000,...,230.80000,0.15559,359.11111,1256.88889,0.61104,0.0,0.0,0.0,589.91111,0.0
27,Large squids,Regular,4.39837,0.29004,0.65522,0.00001,0.17709,4.60000,15.86000,0.81460,...,0.28086,0.15777,0.44311,1.55090,0.24108,0.0,0.0,0.0,0.72398,0.0
26,Small squids,Regular,3.86046,0.30005,0.66673,0.00000,1.15744,5.50000,18.33000,6.36594,...,2.12159,0.15554,3.30000,11.54999,-1.80330,0.0,0.0,0.0,5.42159,0.0
25,Other turtles,Regular,3.50449,0.04285,0.00000,0.00003,0.00077,0.14994,3.49935,0.00012,...,0.00011,0.21270,0.00057,0.00200,-0.00011,0.0,0.0,0.0,0.00069,0.0
24,Leatherback turtle,Regular,3.98591,0.04265,0.00000,0.00001,0.00057,0.14938,3.50264,0.00008,...,0.00008,0.21274,0.00042,0.00148,-0.00008,0.0,0.0,0.0,0.00051,0.0


In [162]:
apply_lim(groups_df.copy(), weight_flow=1.0, weight_guess=1.0).round(8)

,group_name,trophic_info,tl,ge,ee,catch,biomass,pb,qb,p,...,M0,gs,egestion,respiration,biomass_accum,emigration,immigration,net_migration,flow_to_det,detritus_import
group_seq,,,,,,,,,,,,,,,,,,,,,
32,diet_import,Import,1.000000,1.000000,1.000000,0.000000,1.000000,84.566940,84.566940,84.566940,...,0.000000,0.000000,0.000000,0.000000,-0.000000e+00,0.0,0.0,0.0,0.000000,0.0
31,Detritus,DET,1.000000,1.000000,1.000000,0.000000,15.075000,108.962906,108.962906,1642.615813,...,0.000000,0.000000,0.000000,0.000000,4.918374e+02,0.0,0.0,0.0,0.000000,0.0
30,Phytoplankton,PP,1.000000,1.000000,0.890873,0.000000,32.000000,42.799998,42.799998,1369.599930,...,149.459746,0.000000,0.000000,0.000000,6.681297e+01,0.0,0.0,0.0,149.459746,0.0
29,Large zooplankton,Regular,3.005970,0.300000,0.707952,0.000000,9.636000,8.700000,29.000000,83.833200,...,24.483340,0.104193,29.116083,166.494717,4.657130e+01,0.0,0.0,0.0,53.599423,0.0
28,Small zooplankton,Regular,2.003012,0.299827,0.666034,0.000000,40.000000,17.300000,57.700000,692.000000,...,231.104711,0.155593,359.109745,1256.890255,3.063326e-01,0.0,0.0,0.0,590.214457,0.0
27,Large squids,Regular,4.398368,0.290038,0.643162,0.000007,0.177088,4.600001,15.860002,0.814605,...,0.290682,0.105662,0.296763,1.697248,2.312571e-01,0.0,0.0,0.0,0.587445,0.0
26,Small squids,Regular,3.860465,0.300055,0.819044,0.000000,1.157443,5.500000,18.329999,6.365936,...,1.151956,0.104135,2.209312,12.640681,-8.336600e-01,0.0,0.0,0.0,3.361268,0.0
25,Other turtles,Regular,3.504491,0.042847,0.950000,0.000035,0.000767,0.149935,3.499348,0.000115,...,0.000006,0.212701,0.000571,0.001998,-4.750000e-06,0.0,0.0,0.0,0.000577,0.0
24,Leatherback turtle,Regular,3.985911,0.042649,0.950000,0.000006,0.000569,0.149385,3.502636,0.000085,...,0.000004,0.212745,0.000424,0.001484,-4.250000e-06,0.0,0.0,0.0,0.000428,0.0
